In [4]:
! uv pip install openai python-dotenv

Checked 2 packages in 1ms


A dedicated LLM client class. This class will encapsulate all details of interacting with model services, allowing our main logic to focus more on agent construction.

In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv
from typing import List, Dict

# Load environment variables from .env file
load_dotenv()

class BasicAgentsLLM:
    """
    A customized LLM client for the book "Hello Agents".
    It is used to call any service compatible with the OpenAI interface and uses streaming responses by default.
    """
    def __init__(self, model: str | None = None, apiKey: str | None = None, baseUrl: str | None = None, timeout: int | None = None):
        """
        Initialize the client. Prioritize passed parameters; if not provided, load from environment variables.
        """
        self.model = model or os.getenv("OPENAI_MODEL_ID")
        apiKey = apiKey or os.getenv("OPENAI_API_KEY")
        baseUrl = baseUrl or os.getenv("OPENAI_BASE_URL")
        timeout = timeout or int(os.getenv("LLM_TIMEOUT", 60))

        if not all([self.model, apiKey, baseUrl]):
            raise ValueError("Model ID, API key, and service address must be provided or defined in the .env file.")

        self.client = OpenAI(api_key=apiKey, base_url=baseUrl, timeout=timeout)

    def think(self, messages: List[Dict[str, str]], temperature: float = 0) -> str | None:
        """
        Call the large language model to think and return its response.
        """
        print(f"🧠 Calling {self.model} model...")
        try:
            response = self.client.chat.completions.create(
                model=self.model,
                messages=messages,
                temperature=temperature,
                stream=True,
            )

            # Handle streaming response
            print("✅ Large language model response successful:")
            collected_content = []
            for chunk in response:
                content = chunk.choices[0].delta.content or ""
                print(content, end="", flush=True)
                collected_content.append(content)
            print()  # Newline after streaming output ends
            return "".join(collected_content)

        except Exception as e:
            print(f"❌ Error occurred when calling LLM API: {e}")
            return None

# # --- Client Usage Example ---
# if __name__ == '__main__':
#     try:
#         llmClient = HelloAgentsLLM()

#         exampleMessages = [
#             {"role": "system", "content": "You are a helpful assistant that writes Python code."},
#             {"role": "user", "content": "Write a quicksort algorithm"}
#         ]

#         print("--- Calling LLM ---")
#         responseText = llmClient.think(exampleMessages)
#         if responseText:
#             print("\n\n--- Complete Model Response ---")
#             print(responseText)

#     except ValueError as e:
#         print(e)


--- Calling LLM ---
🧠 Calling qwen3-8b model...
✅ Large language model response successful:


Here's a simple and clear implementation of the **Quicksort algorithm** in Python using a **recursive, non-in-place approach**. This version selects the **last element** of the list as the pivot and partitions the list into elements less than, equal to, and greater than the pivot. It then recursively sorts the sublists and combines the results.

---

### ✅ **Quicksort Implementation (Recursive, Non-In-Place)**

```python
def quicksort(arr):
    if len(arr) <= 1:
        return arr  # Base case: list is already sorted
    
    pivot = arr[-1]  # Choose the last element as pivot
    left = [x for x in arr[:-1] if x < pivot]  # Elements less than pivot
    middle = [x for x in arr if x == pivot]    # Elements equal to pivot
    right = [x for x in arr[:-1] if x > pivot]  # Elements greater than pivot
    
    return quicksort(left) + middle + quicksort(right)
```

---

### 📌 **How It Works**

1. 

Our first tool is the search function, which receives a query string and then returns search results.

In [ ]:
from tavily import TavilyClient

def tavily_web_search(
    query: str,
    search_depth: str = "basic",
    include_answer: bool = True,
    max_results: int = 5
) -> str:
    """
    Tavily web search tool for AI Agents.
    Returns clean structured search results or error message.

    Args:
        query: Search question/keyword
        search_depth: "basic" or "advanced"
        include_answer: Let Tavily auto-summarize a direct answer
        max_results: Max search results to return

    Returns:
        Dictionary with answer + raw results, or error string
    """
    # Get API key from environment
    api_key = os.environ.get("TAVILY_API_KEY")
    if not api_key:
        return "Error: TAVILY_API_KEY environment variable not set."

    # Initialize client once
    tavily = TavilyClient(api_key=api_key)

    try:
        # Core Tavily search call
        response = tavily.search(
            query=query,
            search_depth=search_depth, # type: ignore
            include_answer=include_answer,
            max_results=max_results
        )
        if response.get("answer"):
            return response["answer"]
        formatted = [f"- {r['title']}: {r['content']}" for r in response.get("results", [])]
        if formatted:
            return '\n'.join(formatted)

        return "No results found."

    except Exception as e:
        return f"Search error: {str(e)}"

Build a General Tool Executor.

In [ ]:
from typing import Dict, Any

class ToolExecutor:
    """
    A tool executor responsible for managing and executing tools.
    """
    def __init__(self):
        self.tools: Dict[str, Dict[str, Any]] = {}

    def registerTool(self, name: str, description: str, func: callable):
        """
        Register a new tool in the toolbox.
        """
        if name in self.tools:
            print(f"Warning: Tool '{name}' already exists and will be overwritten.")
        self.tools[name] = {"description": description, "func": func}
        print(f"Tool '{name}' registered.")

    def getTool(self, name: str) -> callable:
        """
        Get a tool's execution function by name.
        """
        return self.tools.get(name, {}).get("func")

    def getAvailableTools(self) -> str:
        """
        Get a formatted description string of all available tools.
        """
        return "\n".join([
            f"- {name}: {info['description']}"
            for name, info in self.tools.items()
        ])

# # --- Tool Initialization and Usage Example ---
# if __name__ == '__main__':
#     # 1. Initialize tool executor
#     toolExecutor = ToolExecutor()

#     # 2. Register our practical search tool
#     search_description = "A web search engine. Use this tool when you need to answer questions about current events, facts, and information not found in your knowledge base."
#     toolExecutor.registerTool("Search", search_description, tavily_web_search)

#     # 3. Print available tools
#     print("\n--- Available Tools ---")
#     print(toolExecutor.getAvailableTools())

#     # 4. Agent's Action call, this time we ask a real-time question
#     print("\n--- Execute Action: Search['What is NVIDIA's latest GPU model'] ---")
#     tool_name = "Search"
#     tool_input = "What is NVIDIA's latest GPU model"

#     tool_function = toolExecutor.getTool(tool_name)
#     if tool_function:
#         observation = tool_function(tool_input)
#         print("--- Observation ---")
#         print(observation)
#     else:
#         print(f"Error: Tool named '{tool_name}' not found.")


Tool 'Search' registered.

--- Available Tools ---
- Search: A web search engine. Use this tool when you need to answer questions about current events, facts, and information not found in your knowledge base.

--- Execute Action: Search['What is NVIDIA's latest GPU model'] ---
--- Observation ---
{'query': "What is NVIDIA's latest GPU model", 'follow_up_questions': None, 'answer': "As of 2026, NVIDIA's latest GPU model is the RTX 5080. It features DLSS 4 Multi-Frame technology for enhanced performance.", 'images': [], 'results': [{'url': 'https://www.reddit.com/r/gpu/comments/1ozhr4m/nvidias_next_flagship_gpu_the_rtx_6090_already/', 'title': "Nvidia's next flagship GPU, the RTX 6090, already has a rumored ...", 'content': 'Nvidia reportedly moving forward with 9GB versions of the RTX 5060 & RTX 5060 Ti, with some compromises · RTX 5090 lasts about a minute after', 'score': 0.54782754, 'raw_content': None}, {'url': 'https://www.nvidia.com/en-us/products/workstations/professional-desktop

Assemble all independent components—the LLM client and tool executor—to build a complete ReAct agent. 

![Figure 4.1 Think-Act-Observe Synergistic Loop in ReAct Paradigm | https://github.com/datawhalechina/Hello-Agents ](./docs/4-1.png)

In [ ]:
# ReAct Prompt Template
REACT_PROMPT_TEMPLATE = """
Please note that you are an intelligent assistant capable of calling external tools.

Available tools are as follows:
{tools}

Please respond strictly in the following format:

Thought: Your thinking process, used to analyze problems, decompose tasks, and plan the next action.
Action: The action you decide to take, must be in one of the following formats:
- {{tool_name}}[{{tool_input}}]`: Call an available tool.
- `Finish[final answer]`: When you believe you have obtained the final answer.
- When you have collected enough information to answer the user's final question, you must use `Finish[final answer]` after the Action: field to output the final answer.

Now, please start solving the following problem:
Question: {question}
History: {history}
"""


The core of ReActAgent is a loop that continuously "formats prompt -> calls LLM -> executes action -> integrates results" until the task is complete or the maximum step limit is reached.

In [ ]:
import re
class ReActAgent:
    def __init__(self, llm_client: BasicAgentsLLM, tool_executor: ToolExecutor, max_steps: int = 5):
        self.llm_client = llm_client
        self.tool_executor = tool_executor
        self.max_steps = max_steps
        self.history = []

    def run(self, question: str):
        """
        Run the ReAct agent to answer a question.
        """
        self.history = [] # Reset history for each run
        current_step = 0

        while current_step < self.max_steps:
            current_step += 1
            print(f"--- Step {current_step} ---")

            # 1. Format prompt
            tools_desc = self.tool_executor.getAvailableTools()
            history_str = "\n".join(self.history)
            prompt = REACT_PROMPT_TEMPLATE.format(
                tools=tools_desc,
                question=question,
                history=history_str
            )

            # 2. Call LLM to think
            messages = [{"role": "user", "content": prompt}]
            response_text = self.llm_client.think(messages=messages)

            if not response_text:
                print("Error: LLM failed to return a valid response.")
                break
            # 3. Parse LLM output
            thought, action = self._parse_output(response_text)

            if thought:
                print(f"Thought: {thought}")

            if not action:
                print("Warning: Failed to parse valid Action, process terminated.")
                break

            # 4. Execute Action
            if action.startswith("Finish"):
                # If it's a Finish instruction, extract the final answer and end
                final_answer = re.match(r"Finish\[(.*)\]", action).group(1)
                print(f"🎉 Final Answer: {final_answer}")
                return final_answer

            tool_name, tool_input = self._parse_action(action)
            if not tool_name or not tool_input:
                # ... Handle invalid Action format ...
                continue

            print(f"🎬 Action: {tool_name}[{tool_input}]")

            tool_function = self.tool_executor.getTool(tool_name)
            if not tool_function:
                observation = f"Error: Tool named '{tool_name}' not found."
            else:
                observation = tool_function(tool_input) # Call real tool
                # (This logic follows tool invocation, at the end of the while loop)
            print(f"👀 Observation: {observation}")

            # Add this round's Action and Observation to history
            self.history.append(f"Action: {action}")
            self.history.append(f"Observation: {observation}")

        # Loop ends
        print("Maximum steps reached, process terminated.")
        return None



    def _parse_output(self, text: str):
        """Parse LLM output to extract Thought and Action.
        """
        # Thought: match until Action: or end of text
        thought_match = re.search(r"Thought:\s*(.*?)(?=\nAction:|$)", text, re.DOTALL)
        # Action: match until end of text
        action_match = re.search(r"Action:\s*(.*?)$", text, re.DOTALL)
        thought = thought_match.group(1).strip() if thought_match else None
        action = action_match.group(1).strip() if action_match else None
        return thought, action

    def _parse_action(self, action_text: str):
        """Parse Action string to extract tool name and input.
        """
        match = re.match(r"(\w+)\[(.*)\]", action_text, re.DOTALL)
        if match:
            return match.group(1), match.group(2)
        return None, None


Plan-and-Solve is more like an architect with two major phase:

- Planning Phase: First, decompose the problem into three independent calculation steps (calculate Tuesday sales, calculate Wednesday sales, calculate total sales).
- Execution Phase: Then, strictly follow the plan, execute calculations step by step, and use each step's result as input for the next step, finally obtaining the total.

![Figure 4.2 Two-Stage Workflow of Plan-and-Solve Paradigm | https://github.com/datawhalechina/Hello-Agents](./docs/4-2.png)

The goal of the planning phase is to have the large language model receive the original problem and output a clear, step-by-step action plan. This plan must be structured so our code can easily parse and execute it one by one. 

In [ ]:
PLANNER_PROMPT_TEMPLATE = """
You are a top AI planning expert. Your task is to decompose complex problems posed by users into an action plan consisting of multiple simple steps.
Please ensure that each step in the plan is an independent, executable subtask and is strictly arranged in logical order.
Your output must be a Python list, where each element is a string describing a subtask.

Question: {question}

Please strictly output your plan in the following format, with ```python and ``` as prefix and suffix being necessary:
```python
["Step 1", "Step 2", "Step 3", ...]
```
"""


In [ ]:
import ast

class Planner:
    def __init__(self, llm_client: BasicAgentsLLM):
        self.llm_client = llm_client

    def plan(self, question: str) -> list[str]:
        """
        Generate an action plan based on user question.
        """
        prompt = PLANNER_PROMPT_TEMPLATE.format(question=question)

        # To generate a plan, we build a simple message list
        messages = [{"role": "user", "content": prompt}]

        print("--- Generating Plan ---")
        # Use streaming output to get the complete plan
        response_text = self.llm_client.think(messages=messages) or ""

        print(f"✅ Plan Generated:\n{response_text}")

        # Parse the list string output by LLM
        try:
            # Find content between ```python and ```
            plan_str = response_text.split("```python")[1].split("```")[0].strip()
            # Use ast.literal_eval to safely execute the string and convert it to a Python list
            plan = ast.literal_eval(plan_str)
            return plan if isinstance(plan, list) else []
        except (ValueError, SyntaxError, IndexError) as e:
            print(f"❌ Error parsing plan: {e}")
            print(f"Raw response: {response_text}")
            return []
        except Exception as e:
            print(f"❌ Unknown error occurred while parsing plan: {e}")
            return []


The executor is not only responsible for calling the large language model to solve each sub-problem but also plays a crucial role: state management. It must record the execution results of each step and provide them as context for subsequent steps, ensuring information flows smoothly throughout the entire task chain.

In [ ]:
EXECUTOR_PROMPT_TEMPLATE = """
You are a top AI execution expert. Your task is to strictly follow the given plan and solve the problem step by step.
You will receive the original question, the complete plan, and the steps and results completed so far.
Please focus on solving the "current step" and only output the final answer for that step, without any additional explanations or dialogue.

# Original Question:
{question}

# Complete Plan:
{plan}

# Historical Steps and Results:
{history}

# Current Step:
{current_step}

Please only output the answer for the "current step":
"""


In [ ]:
class Executor:
    def __init__(self, llm_client: BasicAgentsLLM):
        self.llm_client = llm_client

    def execute(self, question: str, plan: list[str]) -> str:
        """
        Execute step by step according to the plan and solve the problem.
        """
        history = "" # String to store historical steps and results

        print("\n--- Executing Plan ---")

        for i, step in enumerate(plan):
            print(f"\n-> Executing step {i+1}/{len(plan)}: {step}")

            prompt = EXECUTOR_PROMPT_TEMPLATE.format(
                question=question,
                plan=plan,
                history=history if history else "None", # If it's the first step, history is empty
                current_step=step
            )

            messages = [{"role": "user", "content": prompt}]

            response_text = self.llm_client.think(messages=messages) or ""

            # Update history for the next step
            history += f"Step {i+1}: {step}\nResult: {response_text}\n\n"

            print(f"✅ Step {i+1} completed, result: {response_text}")

        # After the loop ends, the last step's response is the final answer
        final_answer = response_text
        return final_answer


Now integrate these two components into a unified agent PlanAndSolveAgent and give it complete problem-solving capabilities. 

In [ ]:
class PlanAndSolveAgent:
    def __init__(self, llm_client: BasicAgentsLLM):
        """
        Initialize the agent and create planner and executor instances.
        """
        self.llm_client = llm_client
        self.planner = Planner(self.llm_client)
        self.executor = Executor(self.llm_client)

    def run(self, question: str):
        """
        Run the agent's complete process: plan first, then execute.
        """
        print(f"\n--- Starting to Process Question ---\nQuestion: {question}")

        # 1. Call planner to generate plan
        plan = self.planner.plan(question)

        # Check if plan was successfully generated
        if not plan:
            print("\n--- Task Terminated --- \nUnable to generate valid action plan.")
            return

        # 2. Call executor to execute plan
        final_answer = self.executor.execute(question, plan)

        print(f"\n--- Task Completed ---\nFinal Answer: {final_answer}")


The core idea of the Reflection mechanism is to introduce a post-hoc self-correction loop for the agent, enabling it to review its work, discover deficiencies, and iteratively optimize, just like humans do.

Its core workflow can be summarized as a concise three-step loop: **Execute -> Reflect -> Refine**.

![Figure 4.3 Execute-Reflect-Refine Iterative Loop in Reflection Mechanism | https://github.com/datawhalechina/Hello-Agents](./docs/4-3.png)

The core of Reflection lies in iteration, and the prerequisite for iteration is the ability to remember previous attempts and received feedback. Therefore, a "short-term memory" module is essential for implementing this paradigm. This memory module will be responsible for storing the complete trajectory of each "execute-reflect" loop.

In [ ]:
from typing import List, Dict, Any, Optional, Literal

class Memory:
    """
    A simple short-term memory module for storing the agent's action and reflection trajectory.
    """

    def __init__(self):
        """
        Initialize an empty list to store all records.
        """
        self.records: List[Dict[str, Any]] = []
        # ✅ dedicated cache: NO loops ever
        self._last_execution: Optional[str] = None

    def add_record(self, record_type: Literal["execution", "reflection"], content: str):
        """
        Add a new record to memory.

        Parameters:
        - record_type (str): Type of record ('execution' or 'reflection').
        - content (str): Specific content of the record (e.g., generated code or reflection feedback).
        """
        record = {"type": record_type, "content": content}
        self.records.append(record)
        print(f"📝 Memory updated, added a '{record_type}' record.")
        # Automatically keep last execution up to date
        if record_type == "execution":
            self._last_execution = content

    def get_trajectory(self) -> str:
        """
        Format all memory records into a coherent string text for building prompts.
        """
        trajectory_parts = []
        for record in self.records:
            if record['type'] == 'execution':
                trajectory_parts.append(f"--- Previous Attempt (Code) ---\n{record['content']}")
            elif record['type'] == 'reflection':
                trajectory_parts.append(f"--- Reviewer Feedback ---\n{record['content']}")

        return "\n\n".join(trajectory_parts)

    def get_last_execution(self) -> Optional[str]:
        """
        Get the most recent execution result (e.g., the latest generated code).
        Returns None if it doesn't exist.
        """
        """
        Get the most recent execution result (INSTANT, even for 1M records)
        """
        return self._last_execution


The workflow of `ReflectionAgent` will revolve around the "execute-reflect-refine" loop we discussed earlier and guide the large language model to play different roles through carefully designed prompts.

In [ ]:
INITIAL_PROMPT_TEMPLATE = """
You are a senior Python programmer. Please write a Python function according to the following requirements.
Your code must include a complete function signature, docstring, and follow PEP 8 coding standards.

Requirement: {task}

Please output the code directly without any additional explanations.
"""

REFLECT_PROMPT_TEMPLATE = """
You are an extremely strict code review expert and senior algorithm engineer with ultimate requirements for code performance.
Your task is to review the following Python code and focus on finding its main bottlenecks in <strong>algorithm efficiency</strong>.

# Original Task:
{task}

# Code to Review:
```python
{code}
```

Please analyze the time complexity of this code and consider whether there is an <strong>algorithmically superior</strong> solution to significantly improve performance.
If one exists, please clearly point out the deficiencies of the current algorithm and propose specific, feasible algorithm improvement suggestions (e.g., using sieve method instead of trial division).
Only if the code has reached optimality at the algorithm level can you answer "no improvement needed."

Please output your feedback directly without any additional explanations.
"""


REFINE_PROMPT_TEMPLATE = """
You are a senior Python programmer. You are optimizing your code based on feedback from a code review expert.

# Original Task:
{task}

# Your Previous Code Attempt:
{last_code_attempt}
Reviewer's Feedback:
{feedback}

Please generate an optimized new version of the code based on the reviewer's feedback.
Your code must include a complete function signature, docstring, and follow PEP 8 coding standards.
Please output the optimized code directly without any additional explanations.
"""


In [ ]:
class ReflectionAgent:
    def __init__(self, llm_client: BasicAgentsLLM, max_iterations=3):
        self.llm_client = llm_client
        self.memory = Memory()
        self.max_iterations = max_iterations

    def run(self, task: str):
        """
        Task\n
        ↓\n
        [Initial Code] → starting code\n
        ↓\n
        ┌─────────────────────────────────┐\n
        │      Iteration Loop (x3)        │\n
        │  1. Reflect → critique          │\n
        │  2. Check → stop if perfect     │\n
        │  3. Refine → better code        │\n
        └─────────────────────────────────┘\n
        ↓\n
        Return Final Code
        """
        print(f"\n--- Starting to Process Task ---\nTask: {task}")

        # --- 1. Initial Execution ---
        print("\n--- Performing Initial Attempt ---")
        initial_prompt = INITIAL_PROMPT_TEMPLATE.format(task=task)
        initial_code = self._get_llm_response(initial_prompt)
        self.memory.add_record("execution", initial_code)

        # --- 2. Iterative Loop: Reflection and Refinement ---
        for i in range(self.max_iterations):
            print(f"\n--- Iteration {i+1}/{self.max_iterations} ---")

            # a. Reflection
            print("\n-> Performing Reflection...")
            last_code = self.memory.get_last_execution()
            if not last_code:
                print("\n⚠️ Missing code — restarting generation")
                last_code = self._get_llm_response(INITIAL_PROMPT_TEMPLATE.format(task=task))
                self.memory.add_record("execution", last_code)

            reflect_prompt = REFLECT_PROMPT_TEMPLATE.format(task=task, code=last_code)
            feedback = self._get_llm_response(reflect_prompt)
            if not feedback:
                print("\n❌ Reflection failed — stopping")
                break
            self.memory.add_record("reflection", feedback)

            # b. Check if stopping is needed
            stop_phrases = [
                "no improvement needed",
                "no changes required",
                "code is correct",
                "satisfactory"
            ]
            if any(phrase in feedback.lower() for phrase in stop_phrases):
                break

            # c. Refinement
            print("\n-> Performing Refinement...")
            refine_prompt = REFINE_PROMPT_TEMPLATE.format(
                task=task,
                last_code_attempt=last_code,
                feedback=feedback
            )
            refined_code = self._get_llm_response(refine_prompt)
            # Avoid useless loops
            if refined_code.strip() == last_code.strip():
                print("\n🔄 No changes made — stopping")
                break
            self.memory.add_record("execution", refined_code)

        final_code = self.memory.get_last_execution()
        print(f"\n--- Task Completed ---\nFinal Generated Code:\n```python\n{final_code}\n```")
        return final_code

    def _get_llm_response(self, prompt: str) -> str:
        """A helper method for calling LLM and getting complete streaming response."""
        messages = [{"role": "user", "content": prompt}]
        response_text = self.llm_client.think(messages=messages) or ""
        return response_text
